# AMML Ranking 2026 · Ранжирование

Обучение и инференс выполнял на Linux, Python 3.13.16 и NVIDIA Tesla T4. Здесь воспроизвожу итоговое решение из исходных CSV и публичных весов Qwen.


## 1. Конфигурация

Задаю пути к данным и параметры итогового решения.


In [ ]:
# Пути и время сеанса
CONFIG = {
    'input_root': 'data',
    'train_path': 'data/train.csv',
    'test_path': 'data/test.csv',
    'sample_submission_path': 'data/sample_submission.csv',
    'output_root': 'outputs/',
    'cache_dir': '.cache/amml-qwen4b/',
    'run_id': 'qwen-final-01',
    'max_run_minutes': None,

    # Зафиксированное решение
    'seed': 2026,
    'max_length': 512,
    'train_microbatch': 2,
    'eval_batch': 8,
    'queries_per_update': 4,
    'lora_rank': 8,
    'lora_alpha': 16,
    'lora_dropout': 0.05,
    'learning_rate': 5e-05,
    'weight_decay': 0.01,
    'grad_clip_norm': 1.0,
    'profile_updates': 8,
    'checkpoint_every': 24,
    'inference_time_margin': 1.35,
    'save_reserve_seconds': 180,
    'selected_step': 533,
    'inference_length': 768,
}

REQUIREMENTS = {'torch': '2.11.0+cu130',
 'transformers': '5.20.2',
 'tokenizers': '0.23.3',
 'peft': '0.21.2',
 'accelerate': '1.15.0',
 'safetensors': '0.8.0',
 'huggingface-hub': '1.33.0',
 'bitsandbytes': '0.49.2',
 'regex': '2026.1.15',
 'click': '8.4.2',
 'hf-xet': '1.6.0'}

CODE_SHA256 = 'c1ac90c69082dc5e3941b3b730d60d46ddcaa3c4545556ade0c8ae23129f2ce2'
COMPUTE_SHA256 = '727ae74c18fca71dfab7598fea4a61e944da6fc8d3fb823671839e63a2035fbc'


## 2. Окружение

Подключаю библиотеки и проверяю совместимость среды.


In [ ]:
import csv
import hashlib
import importlib.metadata
import io
import json
import math
import platform
import random
import shutil
import sys
import time
from collections import defaultdict
from contextlib import nullcontext
from fractions import Fraction
from hashlib import sha256
from itertools import chain
from math import ceil, cos, isfinite, pi
from operator import itemgetter
from pathlib import Path
from tempfile import NamedTemporaryFile


In [ ]:
GIB = 1024 ** 3


class BudgetExpired(RuntimeError):
    pass


def require_time(deadline):
    if time.monotonic() >= deadline:
        raise BudgetExpired("Work deadline reached; reserved time is for checkpoint and output.")


def setup_d03_environment(cfg, requirements, deadline, result):
    if sys.platform != "linux" or platform.machine() != "x86_64":
        raise RuntimeError("Для запуска требуется ядро Jupyter на Linux x86_64 с NVIDIA CUDA GPU.")

    def installed(name):
        try:
            return importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            return None

    versions = {name: installed(name) for name in requirements}
    differences = [f"{name}=={pin} (обнаружено {versions[name] or 'отсутствует'})"
                   for name, pin in requirements.items() if versions[name] != pin]
    instructions = (f'В каталоге сдачи выполните "{sys.executable}" -m pip install -r requirements.txt '
                    "и перезапустите выбранное ядро Jupyter.")
    torchao = installed("torchao")
    if torchao and torchao.split("+", 1)[0] == "0.10.0":
        raise RuntimeError(f"Обнаружен конфликт с torchao {torchao}. Создайте чистое окружение "
                           "с requirements.txt и выберите его ядро Jupyter.")
    if differences:
        raise RuntimeError("Расхождения версий: " + ", ".join(differences) + ". " + instructions)

    require_time(deadline)
    try:
        import torch
    except (ImportError, OSError) as exc:
        raise RuntimeError("Ядро не смогло загрузить PyTorch. Проверьте драйвер NVIDIA. "
                           + instructions) from exc

    if str(torch.__version__) != requirements["torch"] or torch.version.cuda != "13.0":
        raise RuntimeError(f"Загружен torch {torch.__version__}, CUDA {torch.version.cuda}. "
                           f"Требуется torch {requirements['torch']} с CUDA 13.0. " + instructions)
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA недоступна в этом ядре. Проверьте драйвер NVIDIA, видимость GPU "
                           "и сборку PyTorch для CUDA 13.0.")
    props = torch.cuda.get_device_properties(0)
    result["environment"] = {
        "os": platform.platform(), "python": platform.python_version(),
        "gpu": props.name, "gpu_bytes": props.total_memory, "device": "cuda:0",
        "visible_cuda_devices": torch.cuda.device_count(), "cuda": torch.version.cuda,
        "compute_capability": f"{props.major}.{props.minor}", "packages": versions,
    }
    if (props.major, props.minor) < (7, 5) or props.total_memory < 14 * GIB:
        raise RuntimeError(f"Требуется cuda:0 с compute capability от 7.5 и памятью от 14 ГиБ. "
                           f"Обнаружено: {props.name}, {props.major}.{props.minor}, "
                           f"{props.total_memory / GIB:.1f} ГиБ.")
    require_time(deadline)
    return torch, "cuda:0"


## 3. Данные и разбиение

Читаю CSV и сохраняю связанные запросы в одной части выборки.


In [ ]:
def resolve_input(explicit, root, filename):
    if explicit:
        path = Path(explicit)
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    matches = sorted(path for path in Path(root).rglob(filename) if path.is_file())
    if len(matches) != 1:
        raise ValueError(f"Expected one {filename} under {root}; found {matches}. Set an explicit path.")
    return matches[0]


def require_space(directory, minimum):
    free = shutil.disk_usage(directory).free
    if free < minimum:
        raise OSError(f"Insufficient free disk at {directory}: {free} bytes, need {minimum}")
    return free


def dump_json(path, payload):
    temp = path.with_suffix(".tmp")
    temp.write_text(json.dumps(payload, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    temp.replace(path)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def batches(rows, size):
    for start in range(0, len(rows), size):
        yield rows[start:start + size]


def read_verified(path, sha256):
    payload = Path(path).read_bytes()
    if hashlib.sha256(payload).hexdigest() != sha256:
        raise ValueError(f"Unexpected SHA-256: {path}")
    return payload


def csv_rows(payload, required):
    reader = csv.DictReader(io.StringIO(payload.decode("utf-8-sig"), newline=""))
    fields = reader.fieldnames or []
    if len(set(fields)) != len(fields) or not set(required).issubset(fields):
        raise ValueError("Invalid CSV header")
    for row in reader:
        if None in row or None in row.values():
            raise ValueError("Wrong number of CSV fields")
        yield row


In [ ]:
VERSION = "amml-stage-a-v1"


QUOTAS = {"fit": 1693, "dev": 456, "control": 456}


FIELDS = ("query_id", "component_id", "split")


def query_components(rows, candidate_key):
    """Group whole queries connected transitively by a shared candidate key."""
    queries_by_candidate = defaultdict(set)
    for row in rows:
        queries_by_candidate[candidate_key(row)].add(row["query_id"])
    neighbors = defaultdict(set)
    for queries in queries_by_candidate.values():
        first = min(queries)
        neighbors[first].update(queries - {first})
        for query_id in queries - {first}:
            neighbors[query_id].add(first)

    unseen = set(neighbors)
    components = []
    for query_id in sorted(neighbors):
        if query_id not in unseen:
            continue
        unseen.remove(query_id)
        component = [query_id]
        for member in component:
            adjacent = neighbors[member] & unseen
            unseen.difference_update(adjacent)
            component.extend(sorted(adjacent))
        components.append(tuple(sorted(component)))
    return components


def allocate(components, quotas):
    # The pinned input has a unique largest component.
    largest = min(components, key=lambda group: (-len(group), group[0]))
    assigned = [(query, largest[0], "fit") for query in largest]
    filled = dict.fromkeys(quotas, 0)

    def order(group):
        component_id = group[0]
        return -len(group), sha256(f"{VERSION}:{component_id}".encode()).digest(), component_id

    for group in sorted((group for group in components if group != largest), key=order):
        eligible = [part for part in quotas if filled[part] + len(group) <= quotas[part]]
        if not eligible:
            remaining = {part: quotas[part] - filled[part] for part in quotas}
            raise ValueError(f"Component {group[0]} of size {len(group)} exceeds remaining capacities {remaining}")
        part = min(eligible, key=lambda name: Fraction(filled[name], quotas[name]))
        filled[part] += len(group)
        assigned.extend((query, group[0], part) for query in group)
    if filled != quotas:
        raise ValueError(f"Unfilled quotas: assigned {filled}, expected {quotas}")
    return sorted(assigned)


def render_assignments(assignments):
    stream = io.StringIO(newline="")
    writer = csv.writer(stream, lineterminator="\n")
    writer.writerow(FIELDS)
    writer.writerows(assignments)
    return stream.getvalue().encode("utf-8")


def build_split_v1_bytes(rows):
    return render_assignments(allocate(query_components(rows, itemgetter("text")), QUOTAS))


def materialize_split(output_root, payload, sha256):
    if hashlib.sha256(payload).hexdigest() != sha256:
        raise ValueError("Embedded query_split_v1.csv differs from the approved SHA-256")
    path = Path(output_root) / "query_split_v1.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        with NamedTemporaryFile(dir=path.parent, delete=False) as handle:
            temporary = Path(handle.name)
            try:
                handle.write(payload)
                handle.close()
                temporary.replace(path)
            finally:
                temporary.unlink(missing_ok=True)
    read_verified(path, sha256)
    return str(path)


def materialize_split_from_train(train_path, output_root, config):
    train = read_verified(train_path, config["train_sha256"])
    rows = list(csv_rows(train, ("query_id", "text")))
    return materialize_split(output_root, build_split_v1_bytes(rows), config["split_sha256"])


In [ ]:
TRAIN_SHA256 = "2c1b26c5d3ee642f2c959b758cea49d4176ca4c0dfe92fbb7236733eff801042"


SPLIT_SHA256 = "7840e07037c3fa3a556c14c68a58edf32286626882bfb79b359214dcf755cac1"


FIT_QUERIES, FIT_ROWS = 2130, 22116


DEV_QUERIES, DEV_ROWS = 456, 4702


TEST_SHA256 = "4667c49296b72bf64767338c5345cb2f6c7cfbd0671364e5b728d9d5bcc2bf20"


SAMPLE_SHA256 = "b2cd50bc4fbaa41cacf04a74d01746c0e9ca391c4e5124822278734700cc3090"


OFFICIAL_TEST_ROWS = 13507


def load_groups(train_path, split_path, config):
    train = read_verified(train_path, config["train_sha256"])
    split = read_verified(split_path, config["split_sha256"])
    assignments = {}
    for row in csv_rows(split, ("query_id", "split")):
        qid = row["query_id"]
        part = row["split"]
        if not qid or qid in assignments or part not in ("fit", "dev", "control"):
            raise ValueError("Invalid or duplicate query assignment")
        assignments[qid] = part

    seen_ids = set()
    seen_queries = set()
    groups = {"fit": {}, "dev": {}}
    for row in csv_rows(train, ("id", "query_id", "query", "title", "text", "relevance")):
        row_id, qid = row["id"], row["query_id"]
        if not row_id or row_id in seen_ids or qid not in assignments:
            raise ValueError("Invalid train ID, duplicate ID or unassigned query")
        seen_ids.add(row_id)
        seen_queries.add(qid)
        part = assignments[qid]
        if part == "control":
            continue
        if row["relevance"] not in ("0", "1"):
            raise ValueError("Expected binary relevance")
        groups[part].setdefault(qid, []).append(row)
    if seen_queries != assignments.keys():
        raise ValueError("Split contains query IDs absent from train")
    return groups


def checked_test_inputs(test_path, sample_path):
    test = list(csv_rows(read_verified(test_path, TEST_SHA256), ("id", "query", "title", "text")))
    sample_csv = csv.DictReader(io.StringIO(read_verified(sample_path, SAMPLE_SHA256).decode("utf-8-sig"), newline=""))
    if sample_csv.fieldnames != ["id", "prediction"]:
        raise ValueError("Expected id,prediction in sample_submission.csv")
    sample_ids = [row["id"] for row in sample_csv]
    test_ids = [row["id"] for row in test]
    if len(test_ids) != OFFICIAL_TEST_ROWS or len(set(test_ids)) != OFFICIAL_TEST_ROWS or any(not item.strip() for item in test_ids):
        raise ValueError("Unexpected official test ID count or empty/duplicate ID")
    if len(sample_ids) != OFFICIAL_TEST_ROWS or set(sample_ids) != set(test_ids):
        raise ValueError("Sample submission does not cover official test IDs exactly once")
    return test, sample_ids


def load_d03_inputs(cfg, artifacts):
    root = cfg["input_root"]
    train = resolve_input(cfg["train_path"], root, "train.csv")
    test = resolve_input(cfg["test_path"], root, "test.csv")
    sample = resolve_input(cfg["sample_submission_path"], root, "sample_submission.csv")
    fingerprints = {"train_sha256": TRAIN_SHA256, "split_sha256": SPLIT_SHA256,
                    "test_sha256": TEST_SHA256, "sample_submission_sha256": SAMPLE_SHA256,
                    "split_version": VERSION}
    split = materialize_split_from_train(train, artifacts, fingerprints)
    groups = load_groups(train, split, fingerprints)
    fit = groups["fit"]
    dev = list(chain.from_iterable(groups["dev"].values()))
    if (len(fit), sum(map(len, fit.values())), len(groups["dev"]), len(dev)) != (
        FIT_QUERIES, FIT_ROWS, DEV_QUERIES, DEV_ROWS
    ):
        raise ValueError("Unexpected fit/dev counts for v1")
    test, sample_ids = checked_test_inputs(test, sample)
    return fit, dev, test, sample_ids, fingerprints


## 4. Подготовка текста

Объединяю запрос, заголовок и фрагмент; обрезаю только хвост документа.


In [ ]:
TITLE_SEPARATOR = "\n\n"


def pair_parts(row):
    query = row["query"].strip()
    title = row["title"].strip()
    text = row["text"].strip()
    if not query or not text:
        raise ValueError("Expected a nonempty query and passage")
    return query, "" if title == "NULL" else title, text


def model_pair(row):
    query, title, text = pair_parts(row)
    return query, title + TITLE_SEPARATOR + text if title else text


PREFIX = ('<|im_start|>system\nJudge whether the Document meets the requirements based on '
          'the Query and the Instruct provided. Note that the answer can only be "yes" or '
          '"no".<|im_end|>\n<|im_start|>user\n')


SUFFIX = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"


INSTRUCTION = "Given a web search query, retrieve relevant passages that answer the query"


class QwenInputs:
    def __init__(self, tokenizer, max_length, answer_ids):
        if not tokenizer.is_fast or tokenizer.pad_token_id is None:
            raise ValueError("Expected a fast Qwen tokenizer and a pad token")
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.pad = tokenizer.pad_token_id
        self.prefix = tokenizer.encode(PREFIX, add_special_tokens=False)
        self.suffix = tokenizer.encode(SUFFIX, add_special_tokens=False)
        self.answer_ids = tuple(tokenizer.convert_tokens_to_ids(word) for word in ("yes", "no"))
        if self.answer_ids != tuple(answer_ids) or self.pad in self.answer_ids:
            raise ValueError(f"Unexpected yes/no token IDs: {self.answer_ids}")
        for word, token in zip(("yes", "no"), self.answer_ids):
            if tokenizer.encode(word, add_special_tokens=False) != [token]:
                raise ValueError(f"Answer {word!r} must be a single token")
        self.room = max_length - len(self.prefix) - len(self.suffix)

    def encode(self, rows, deadline=float("inf")):
        encoded, lengths = {}, []
        for chunk in batches(rows, 192):
            require_time(deadline)
            stems, prompts = [], []
            for row in chunk:
                query, document = model_pair(row)
                stem = f"<Instruct>: {INSTRUCTION}\n<Query>: {query}\n<Document>:"
                stems.append(stem)
                prompts.append(stem + " " + document)
            pack = self.tokenizer(prompts, add_special_tokens=False, return_offsets_mapping=True,
                                  return_attention_mask=False, truncation=False, verbose=False)
            for row, stem, tokens, offsets in zip(chunk, stems, pack["input_ids"], pack["offset_mapping"]):
                # Keep the exact official BPE sequence. Trim only its document tail.
                if not any(end > len(stem) + 1 for _, end in offsets[:self.room]) or self.room <= 0:
                    raise ValueError(f"No document tokens fit after complete query: {row['id']}")
                if any(start < len(stem) for start, _ in offsets[self.room:]):
                    raise ValueError(f"Truncation would remove query or document marker: {row['id']}")
                lengths.append(len(self.prefix) + len(tokens) + len(self.suffix))
                encoded[row["id"]] = self.prefix + tokens[:self.room] + self.suffix
        ordered = sorted(lengths)
        report = {f"p{p}": ordered[ceil(p * len(ordered) / 100) - 1] for p in (50, 90, 95, 99, 100)}
        report.update(rows=len(lengths), truncated=sum(length > self.max_length for length in lengths),
                      limit=self.max_length, prefix_tokens=len(self.prefix), suffix_tokens=len(self.suffix))
        return encoded, report

    def collate(self, chunk, torch, device):
        if any(ids[-len(self.suffix):] != self.suffix for ids in chunk):
            raise ValueError("Missing Qwen prompt suffix")
        width = ceil(max(map(len, chunk)) / 8) * 8
        return {key: torch.tensor(value, dtype=torch.long).to(device) for key, value in {
            "input_ids": [[self.pad] * (width - len(ids)) + ids for ids in chunk],
            "attention_mask": [[0] * (width - len(ids)) + [1] * len(ids) for ids in chunk],
        }.items()}


## 5. Модель

Загружаю публичные веса Qwen и добавляю обучаемые LoRA-адаптеры.


In [ ]:
MODEL_ID = "Qwen/Qwen3-Reranker-4B"


REVISION = "22e683669bc0f0bd69640a1354a6d0aebcfeede5"


QWEN_SMALL_FILES = {
    "config.json": "38bff5eac700032a185745e4076eccad7aa453473cafc2a27de412cdb7b79e19",
    "tokenizer_config.json": "253153d0738ceb4c668d2eff957714dd2bea0b56de772a9fdccd96cbf517e6a0",
    "tokenizer.json": "aeb13307a71acd8fe81861d94ad54ab689df773318809eed3cbe794b4492dae4",
}


def load_qwen_d03(cfg, cache, torch):
    from bitsandbytes.nn import Linear4bit
    from huggingface_hub import hf_hub_download
    from peft import LoraConfig, get_peft_model
    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    shared = {"revision": REVISION, "cache_dir": str(cache), "trust_remote_code": False, "token": False}
    for name, sha in QWEN_SMALL_FILES.items():
        path = Path(hf_hub_download(MODEL_ID, name, revision=REVISION, cache_dir=str(cache), token=False))
        if path.parent.name != REVISION or file_sha256(path) != sha:
            raise RuntimeError(f"Pinned file differs: {name}")
    config = AutoConfig.from_pretrained(path.parent, local_files_only=True, trust_remote_code=False)
    if config.model_type != "qwen3" or config.architectures != ["Qwen3ForCausalLM"]:
        raise RuntimeError("Pinned model configuration is not the expected Qwen3 reranker")
    tokenizer = AutoTokenizer.from_pretrained(path.parent, local_files_only=True, use_fast=True,
                                              padding_side="left", trust_remote_code=False)
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, config=config, quantization_config=quant,
                                                 dtype=torch.float16, attn_implementation="sdpa",
                                                 device_map={"": 0}, use_safetensors=True, **shared)
    if not model.is_loaded_in_4bit or not isinstance(model.model.layers[0].self_attn.q_proj, Linear4bit):
        raise RuntimeError("NF4 base failed to load as expected")
    # Freeze directly so the tied 151k x 2560 embedding/output matrix stays FP16.
    # Non-reentrant checkpointing supports frozen embedding inputs without a hook.
    model.requires_grad_(False)
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model = get_peft_model(model, LoraConfig(
        task_type="CAUSAL_LM", r=cfg["lora_rank"], lora_alpha=cfg["lora_alpha"],
        lora_dropout=cfg["lora_dropout"], bias="none", target_modules=["q_proj", "v_proj"],
        revision=REVISION,
    ))
    return model, tokenizer


def validate_target(model, runtime):
    """Public pinned base, FP16/NF4 and the exact 36-layer q/v LoRA topology."""
    from bitsandbytes.nn import Linear4bit
    if not model.is_loaded_in_4bit: raise ValueError('Expected NF4 base')
    quant = model.config.quantization_config
    if not (quant.load_in_4bit and quant.bnb_4bit_quant_type == 'nf4' and quant.bnb_4bit_use_double_quant
            and str(quant.bnb_4bit_compute_dtype) == 'torch.float16'):
        raise ValueError('Quantization/precision differs')
    expected = {}
    for layer in range(36):
        for projection, width in [('q_proj',4096),('v_proj',1024)]:
            for side,shape in [('A',(8,2560)),('B',(width,8))]:
                expected[f'base_model.model.model.layers.{layer}.self_attn.{projection}.lora_{side}.default.weight'] = shape
    if {n:tuple(p.shape) for n,p in runtime[0].items()} != expected: raise ValueError('Expected exact 144-tensor Qwen4B LoRA topology')
    if any(str(p.dtype) != 'torch.float32' for p in runtime[0].values()): raise ValueError('LoRA precision differs')
    base = model.get_base_model()
    if not isinstance(base.model.layers[0].self_attn.q_proj.base_layer,Linear4bit): raise ValueError('NF4 layer missing')
    if str(base.get_input_embeddings().weight.dtype) != 'torch.float16': raise ValueError('Embedding precision differs')
    if not model.is_gradient_checkpointing: raise ValueError('Gradient checkpointing required')


## 6. Оценка релевантности

Рассчитываю score как разность логитов ответов yes и no.


In [ ]:
def qwen_score(model, encoded, answer_ids, torch, device, full=False):
    # Every row is left-padded, thus index -1 is the last unmasked token.
    amp = torch.autocast("cuda", dtype=torch.float16) if device.startswith("cuda") else nullcontext()
    with amp:
        logits = model(**encoded, use_cache=False, logits_to_keep=0 if full else 1).logits
    expected = encoded["input_ids"].shape[1] if full else 1
    if logits.shape[:2] != (encoded["input_ids"].shape[0], expected):
        raise RuntimeError(f"Unexpected Qwen logits shape: {tuple(logits.shape)}")
    yes, no = answer_ids
    return logits[:, -1, yes].float() - logits[:, -1, no].float()


def checked_score_path(model, inputs, examples, torch, device):
    model.eval()  # adapter dropout off for deterministic comparison, still with autograd
    items = [x[0] for x in examples]
    feed = inputs.collate(items, torch, device)
    mask = feed["attention_mask"]
    if (not bool(mask[:, -1].all()) or not bool((mask[:, 1:] >= mask[:, :-1]).all())
            or not bool((mask == 0).any()) or mask.sum(dim=1).tolist() != list(map(len, items))
            or not bool((feed["input_ids"][mask == 0] == inputs.pad).all())):
        raise RuntimeError("Unexpected left padding or attention mask")
    y = torch.tensor([x[1] for x in examples], dtype=torch.float32, device=device)

    def scores_and_grads(full):
        model.zero_grad(set_to_none=True)
        scores = qwen_score(model, feed, inputs.answer_ids, torch, device, full=full)
        bce = torch.nn.functional.binary_cross_entropy_with_logits(scores, y)
        if not bool(torch.isfinite(bce)):
            raise RuntimeError("Non-finite score or label loss in probe")
        # Compare the scoring Jacobian even if the official model saturates BCE here.
        (scores.mean() * 16).backward()
        grads = torch.cat([p.grad.detach().flatten() for p in model.parameters() if p.requires_grad])
        return scores.detach(), grads

    ref, ref_grad = scores_and_grads(True)
    opt, opt_grad = scores_and_grads(False)
    model.zero_grad(set_to_none=True)
    atol, rtol = (0.10, 0.008) if device.startswith("cuda") else (2e-5, 2e-5)
    torch.testing.assert_close(opt, ref, atol=atol, rtol=rtol)
    with torch.no_grad():
        solo = torch.cat([qwen_score(model, inputs.collate([x], torch, device), inputs.answer_ids,
                                     torch, device) for x in items])
        torch.testing.assert_close(opt, solo, atol=atol, rtol=rtol)
    norm = ref_grad.norm().item()
    rel_error = (ref_grad - opt_grad).norm().item() / max(norm, 1e-20)
    if not isfinite(norm) or not isfinite(rel_error) or norm == 0 or rel_error > (0.10 if device.startswith("cuda") else 2e-4):
        raise RuntimeError(f"Qwen logits_to_keep adapter gradient mismatch: relative L2={rel_error}, norm={norm}")
    return {"max_score_error": (opt - ref).abs().max().item(),
            "max_padding_score_error": (opt - solo).abs().max().item(),
            "relative_adapter_gradient_error": rel_error, "reference_adapter_gradient_norm": norm}


## 7. Обучение

Готовлю фиксированный план обновлений, функцию потерь и сохранение состояния.


In [ ]:
HORIZON = 533


PLAN_SHA = '555d6afa2d6a0f21ff2d0568a5335c0fd2879f0628f283fcab7cd94f66846861'


def token_work(batch, encoded):
    width = 8 * ceil(max(len(encoded[row["id"]]) for row in batch) / 8)
    return width * len(batch)


def make_fit_plan(groups, encoded, seed, queries_per_update):
    ids = sorted(groups)
    random.Random(seed).shuffle(ids)
    plan = []
    for qids in batches(ids, queries_per_update):
        rows = list(chain.from_iterable(groups[qid] for qid in qids))
        # Sorting only within an update cuts padding without changing its objective.
        rows.sort(key=lambda row: (len(encoded[row["id"]]), row["id"]))
        plan.append((qids, rows))
    return plan


def make_work_plan(fit, fit_enc, cfg):
    plan = make_fit_plan(fit, fit_enc, cfg["seed"], cfg["queries_per_update"])
    # Put one real, expensive update first to test memory with full fit queries.
    work = [sum(token_work(chunk, fit_enc) for chunk in batches(rows, cfg["train_microbatch"]))
            for _, rows in plan]
    stress = max(range(len(plan)), key=work.__getitem__)
    plan.insert(0, plan.pop(stress))
    work.insert(0, work.pop(stress))
    return plan, work


def plan_digest(plan):
    return hashlib.sha256('\n'.join('\t'.join(r['id'] for r in rs) for _, rs in plan).encode()).hexdigest()


In [ ]:
def schedule_multiplier(step, total):
    warmup = max(1, ceil(0.04 * total))
    if step < warmup:
        return (step + 1) / warmup
    progress = min(1, (step - warmup) / max(1, total - warmup - 1))
    return 0.35 + 0.65 * (1 + cos(pi * progress)) / 2


def build_d03_optimizer(model, total, cfg, torch, device):
    trainable = {name: p for name, p in model.named_parameters() if p.requires_grad}
    if not trainable or any("lora_" not in name or p.dtype != torch.float32 for name, p in trainable.items()):
        raise RuntimeError("Expected only FP32 LoRA parameters to be trainable")
    optimizer = torch.optim.AdamW(list(trainable.values()), lr=cfg["learning_rate"],
                                  weight_decay=cfg["weight_decay"], fused=False)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: schedule_multiplier(step, total))
    scaler = torch.amp.GradScaler("cuda", enabled=device.startswith("cuda"), init_scale=128.0)
    return trainable, optimizer, scheduler, scaler


def fit_update(model, rows, encoded, inputs, runtime, cfg, torch, device, deadline):
    trainable, optimizer, scheduler, scaler = runtime
    model.train()
    size = len(rows)
    for attempt in range(5):
        optimizer.zero_grad(set_to_none=True)
        losses = []
        for chunk in batches(rows, cfg["train_microbatch"]):
            require_time(deadline)
            feed = inputs.collate([encoded[row["id"]] for row in chunk], torch, device)
            y = torch.tensor([int(row["relevance"]) for row in chunk], device=device, dtype=torch.float32)
            scores = qwen_score(model, feed, inputs.answer_ids, torch, device)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(scores, y, reduction="sum") / size
            scaler.scale(loss).backward()
            losses.append(loss.detach())
        avg = torch.stack(losses).sum().item()
        if not isfinite(avg) or any(p.grad is None for p in trainable.values()):
            raise RuntimeError("Non-finite fit loss or missing LoRA gradient")
        scaler.unscale_(optimizer)
        norm = torch.nn.utils.clip_grad_norm_(list(trainable.values()), cfg["grad_clip_norm"]).item()
        if not isfinite(norm):
            if not scaler.is_enabled():
                raise RuntimeError("Non-finite unscaled LoRA gradient")
            scaler.update(new_scale=scaler.get_scale() * 0.5)
            continue
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        return {"loss": avg, "grad_norm": norm, "amp_retries": attempt}
    raise RuntimeError("Five consecutive FP16 overflow attempts on the same fit update")


In [ ]:
def capture_rng(torch, device="cuda"):
    state = {"python": random.getstate(), "cpu": torch.get_rng_state()}
    if device == "cuda":
        state["cuda"] = torch.cuda.get_rng_state(0)
    return state


def restore_rng(state, torch, device="cuda"):
    random.setstate(state["python"])
    torch.set_rng_state(state["cpu"])
    if device == "cuda":
        torch.cuda.set_rng_state(state["cuda"], 0)


def save_d03_checkpoint(path, model_state, identity, step, progress, torch, device):
    trainable, optimizer, scheduler, scaler = model_state
    require_space(path.parent, GIB // 2)
    tmp = path.with_suffix(".tmp")
    try:
        torch.save({
            "identity": identity, "step": step, "progress": progress,
            "trainable": {name: p.detach().cpu() for name, p in trainable.items()},
            "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(), "rng": capture_rng(torch, device.split(":")[0]),
        }, tmp)
        tmp.replace(path)
    finally:
        tmp.unlink(missing_ok=True)


def restore_d03_checkpoint(path, model_state, identity, plan, torch, device):
    trainable, optimizer, scheduler, scaler = model_state
    state = torch.load(path, map_location="cpu", weights_only=True)
    step = state["step"]
    if state["identity"] != identity or not 0 < step <= len(plan):
        raise ValueError("Recovery does not belong to this exact code, data, stack, GPU and fit plan")
    want = {"rows": sum(len(rows) for _, rows in plan[:step]),
            "queries": sum(len(qids) for qids, _ in plan[:step])}
    if state["progress"] != want or state["scheduler"]["last_epoch"] != step:
        raise ValueError("Inconsistent recovery cursor")
    tensors = state["trainable"]
    if tensors.keys() != trainable.keys():
        raise ValueError("Inconsistent recovery LoRA keys")
    with torch.no_grad():
        for name, p in trainable.items():
            value = tensors[name]
            if value.shape != p.shape or value.dtype != p.dtype or not bool(torch.isfinite(value).all()):
                raise ValueError(f"Invalid checkpoint tensor: {name}")
            p.copy_(value)
    optimizer.load_state_dict(state["optimizer"])
    if len(optimizer.state) != len(trainable) or any(int(v["step"].item()) != step for v in optimizer.state.values()):
        raise ValueError("Recovery AdamW steps do not match its cursor")
    scheduler.load_state_dict(state["scheduler"])
    scaler.load_state_dict(state["scaler"])
    restore_rng(state["rng"], torch, device.split(":")[0])
    return step, want


def trained_progress(step, progress, total, total_rows):
    if step <= 0:
        raise RuntimeError("A zero-step checkpoint cannot produce the trained submission")
    return dict(progress, steps=step, planned_steps=total, fit_fraction=progress["rows"] / total_rows,
                complete_epoch=step == total)


In [ ]:
def exact_tree(left, right, torch, label='state'):
    if torch.is_tensor(left):
        if not torch.is_tensor(right) or left.dtype != right.dtype or left.shape != right.shape or not torch.equal(left.cpu(), right.cpu()):
            raise ValueError(f'{label}: tensors differ')
    elif isinstance(left, dict):
        if not isinstance(right, dict) or left.keys() != right.keys(): raise ValueError(f'{label}: keys differ')
        for key in left: exact_tree(left[key], right[key], torch, f'{label}.{key}')
    elif isinstance(left, (tuple,list)):
        if type(left) is not type(right) or len(left) != len(right): raise ValueError(f'{label}: sequences differ')
        for i, (a,b) in enumerate(zip(left,right)): exact_tree(a,b,torch,f'{label}.{i}')
    elif left != right:
        raise ValueError(f'{label}: {left!r} != {right!r}')


def parameter_digest(runtime):
    import torch
    stream=io.BytesIO()
    # Stable ordered serialization on the pinned torch stack; no new numpy dependency.
    torch.save({name:value.detach().cpu().contiguous() for name,value in runtime[0].items()},stream)
    return hashlib.sha256(stream.getbuffer()).hexdigest()


def validate_train_state(state, runtime, plan, cfg, torch, device):
    """Validate moments, cursor and schedule BEFORE mutating live model state."""
    expected_keys = {'identity','step','progress','trainable','optimizer','scheduler','scaler','rng'}
    if set(state) != expected_keys: raise ValueError('Unexpected full checkpoint schema')
    step = state['step']
    if type(step) is not int or not 0 < step <= len(plan): raise ValueError('Invalid cursor')
    want = dict(rows=sum(len(rs) for _,rs in plan[:step]), queries=sum(len(qs) for qs,_ in plan[:step]))
    if state['progress'] != want: raise ValueError('Invalid processed rows/queries')
    trainable, optimizer, scheduler, scaler = runtime
    if list(state['trainable']) != list(trainable): raise ValueError('LoRA topology/order differs')
    for name, value in state['trainable'].items():
        p = trainable[name]
        if value.shape != p.shape or value.dtype != torch.float32 or not bool(torch.isfinite(value).all()):
            raise ValueError(f'Invalid LoRA tensor {name}')
    saved = state['optimizer']; fresh = optimizer.state_dict()
    if len(saved['param_groups']) != 1 or len(fresh['param_groups']) != 1: raise ValueError('Expected one AdamW group')
    group = saved['param_groups'][0]; current = fresh['param_groups'][0]
    if group.keys() != current.keys(): raise ValueError('AdamW group schema differs')
    for key in current:
        if key != 'lr' and group[key] != current[key]: raise ValueError(f'AdamW setting differs: {key}')
    expected_lr = cfg['learning_rate'] * schedule_multiplier(step, len(plan))
    if not math.isclose(group['lr'], expected_lr, rel_tol=1e-12, abs_tol=1e-15): raise ValueError('Actual LR differs from fixed horizon')
    if set(saved['state']) != set(group['params']): raise ValueError('Missing AdamW moments')
    for index, parameter in zip(group['params'], trainable.values()):
        moments = saved['state'][index]
        if set(moments) != {'step','exp_avg','exp_avg_sq'}: raise ValueError('AdamW state schema differs')
        if moments['step'].numel() != 1 or float(moments['step']) != step: raise ValueError('AdamW counter differs')
        for key in ('exp_avg','exp_avg_sq'):
            value = moments[key]
            if value.shape != parameter.shape or value.dtype != parameter.dtype or not bool(torch.isfinite(value).all()):
                raise ValueError('Invalid AdamW moment')
        if bool((moments['exp_avg_sq'] < 0).any()): raise ValueError('Negative second moment')
    sched = state['scheduler']
    expected_sched = scheduler.state_dict()
    expected_sched.update(last_epoch=step, _step_count=step+1, _last_lr=[expected_lr])
    exact_tree(sched, expected_sched, torch, 'scheduler')
    scale = state['scaler']
    if scaler.is_enabled():
        if set(scale) != set(scaler.state_dict()) or not math.isfinite(scale['scale']) or scale['scale'] <= 0:
            raise ValueError('Invalid scaler')
        for key in ('growth_factor','backoff_factor','growth_interval'):
            if scale[key] != scaler.state_dict()[key]: raise ValueError('Scaler setting differs')
        if not 0 <= scale['_growth_tracker'] < scale['growth_interval']: raise ValueError('Invalid scaler cursor')
    elif scale: raise ValueError('Unexpected enabled scaler on CPU')
    rng = state['rng']; keys = {'python','cpu','cuda'} if device.startswith('cuda') else {'python','cpu'}
    if set(rng) != keys: raise ValueError('Missing or incompatible RNG streams')
    random.Random().setstate(rng['python'])
    for key in keys-{'python'}:
        if rng[key].dtype != torch.uint8 or rng[key].ndim != 1: raise ValueError('Invalid RNG tensor')
    return want


def restore_verified(path, runtime, identity, plan, cfg, torch, device):
    state = torch.load(path, map_location='cpu', weights_only=True)
    if state['identity'] != identity: raise ValueError('Strict resume identity mismatch')
    validate_train_state(state,runtime,plan,cfg,torch,device)
    result = restore_d03_checkpoint(path,runtime,identity,plan,torch,device)
    runtime[1].zero_grad(set_to_none=True)
    for key, actual in [('trainable',runtime[0]),('optimizer',runtime[1].state_dict()),
                        ('scheduler',runtime[2].state_dict()),('scaler',runtime[3].state_dict()),
                        ('rng',capture_rng(torch,device.split(':')[0]))]:
        exact_tree(state[key],actual,torch,key)
    return result


## 8. Пакетная обработка

Группирую пары по длине для экономии памяти.


In [ ]:
def eval_batches(rows, encoded, size):
    return list(batches(sorted(rows, key=lambda r: (len(encoded[r["id"]]), r["id"])), size))


def predict_batches(model, chunks, encoded, inputs, torch, device, deadline):
    model.eval()
    started = time.monotonic()
    scores = []
    with torch.no_grad():
        for chunk in chunks:
            require_time(deadline)
            feed = inputs.collate([encoded[row["id"]] for row in chunk], torch, device)
            pred = qwen_score(model, feed, inputs.answer_ids, torch, device).tolist()
            if not all(map(isfinite, pred)):
                raise RuntimeError("Non-finite inference score")
            scores.extend(zip((row["id"] for row in chunk), pred))
    require_time(deadline)
    return scores, time.monotonic() - started


def workload(chunks, encoded):
    return (len(chunks), sum(token_work(chunk, encoded) for chunk in chunks),
            sum(token_work(chunk, encoded) ** 2 / len(chunk) for chunk in chunks))


def projected_seconds(seconds, observed, target):
    return seconds * max(full / sample for sample, full in zip(observed, target))


def fit_cost(samples, future_work):
    recent = samples[-16:]
    rate = sum(seconds for seconds, _ in recent) / sum(tokens for _, tokens in recent)
    return sum(future_work) * rate


def peak_memory(torch, device):
    if not device.startswith("cuda"):
        return {}
    return {"peak_cuda_allocated_bytes": torch.cuda.max_memory_allocated(0),
            "peak_cuda_reserved_bytes": torch.cuda.max_memory_reserved(0)}


def profile_queries(rows, encoded, count=32):
    """Select length-stratified complete groups, including the longest group."""
    gs = defaultdict(list)
    for r in rows: gs[r['query_id']].append(r)
    ids = sorted(gs, key=lambda q:(max(len(encoded[r['id']]) for r in gs[q]),
                                  sum(len(encoded[r['id']]) for r in gs[q]), q))
    count = min(count, len(ids))
    indices = {round(i * (len(ids)-1) / max(1,count-1)) for i in range(count)}
    return [r for i in sorted(indices) for r in gs[ids[i]]]


## 9. Выполнение обучения

Объединяю подготовку модели и обучение до 533 обновлений.


In [ ]:
OPERATIONAL = {'run_id', 'output_root', 'input_root', 'train_path', 'test_path',
               'sample_submission_path', 'cache_dir', 'max_run_minutes'}


TRAIN_SETTINGS = {'seed': 2026,
 'max_length': 512,
 'train_microbatch': 2,
 'eval_batch': 8,
 'queries_per_update': 4,
 'lora_rank': 8,
 'lora_alpha': 16,
 'lora_dropout': 0.05,
 'learning_rate': 5e-05,
 'weight_decay': 0.01,
 'grad_clip_norm': 1.0}


def validate_config(cfg):
    for key, expected in TRAIN_SETTINGS.items():
        if cfg[key] != expected:
            raise ValueError(f"Изменён параметр обучения {key}: ожидается {expected}")
    if (cfg["selected_step"], cfg["inference_length"]) != (HORIZON, 768):
        raise ValueError("Финальная конфигурация требует полного плана и инференса при 768 токенах")
    minutes = cfg["max_run_minutes"]
    if minutes is not None and (type(minutes) not in (int, float) or minutes <= 0
                                or (isinstance(minutes, float) and not math.isfinite(minutes))):
        raise ValueError("max_run_minutes должен быть None или положительным конечным числом минут")
    if cfg["save_reserve_seconds"] < 180 or cfg["inference_time_margin"] < 1.35:
        raise ValueError("Недостаточный запас на инференс и запись состояния")


def execution_identity(cfg, code, compute, fingerprints, environment, plan, derived_from=None):
    return dict(format='amml-d06-full-state-v1',component='qwen',execution_code_sha256=code,
        compute_sha256=compute,model_id=MODEL_ID,revision=REVISION,**fingerprints,
        environment=environment,configuration={k:v for k,v in cfg.items() if k not in OPERATIONAL},
        plan_sha256=plan_digest(plan),horizon=len(plan),derived_from=derived_from)


def transfer_manifest(out, identity):
    files={str(p.relative_to(out)):file_sha256(p) for p in sorted(out.rglob('*'))
           if p.is_file() and p.name not in ('manifest.json',) and p.suffix != '.tmp'}
    dump_json(out/'manifest.json',dict(format='amml-d06-transfer-v1',identity=identity,files=files))


def new_run(cfg, code, compute):
    validate_config(cfg)
    run_id = cfg["run_id"]
    if not run_id or Path(run_id).name != run_id or run_id in (".", ".."):
        raise ValueError("Укажите имя нового каталога в run_id.")
    out = Path(cfg["output_root"]) / run_id
    try:
        out.mkdir(parents=True, exist_ok=False)
    except FileExistsError:
        raise FileExistsError(f"Каталог {out} уже существует. Для нового запуска измените run_id.") from None
    (out / "artifacts").mkdir()
    require_space(out, GIB)
    started = time.monotonic()
    minutes = cfg["max_run_minutes"]
    deadline = math.inf
    if minutes is not None:
        # Monotonic timestamps are floats, including for integer limits.
        deadline = started + 60 * minutes if minutes < sys.float_info.max / 60 else sys.float_info.max
    summary = dict(run_id=run_id, status="running", configuration=cfg, execution_code_sha256=code,
                   compute_sha256=compute, control_used=False)

    def report(phase, **updates):
        summary.update(phase=phase, elapsed_s=time.monotonic() - started, **updates)
        dump_json(out / "run_summary.json", summary)
        with (out / "events.jsonl").open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(dict(phase=phase, **updates), allow_nan=False) + "\n")
        print(phase, updates.get("step", ""), flush=True)

    report("starting")
    return out, deadline, summary, report


def enough_time(deadline, seconds, phase):
    if deadline-time.monotonic() < seconds: raise BudgetExpired(f'Insufficient measured budget for {phase}: need {seconds:.0f}s')


In [ ]:
def prepare_runtime(cfg, out, deadline, summary, requirements, lengths, environment_validator=None):
    fit,dev,test,sample,fingerprints=load_d03_inputs(cfg,out/'artifacts')
    torch,device=setup_d03_environment(cfg,requirements,deadline,summary)
    if environment_validator: environment_validator(summary['environment'])
    torch.cuda.set_device(0); torch.cuda.reset_peak_memory_stats(0)
    random.seed(cfg['seed']); torch.manual_seed(cfg['seed'])
    cache=Path(cfg['cache_dir']); cache.mkdir(parents=True,exist_ok=True); require_space(cache,12*GIB)
    model,tokenizer=load_qwen_d03(cfg,cache,torch)
    train_inputs=QwenInputs(tokenizer,512,(9693,2152))
    fit_rows=list(chain.from_iterable(fit.values()))
    fit_enc,fit_lengths=train_inputs.encode(fit_rows,deadline)
    plan,work=make_work_plan(fit,fit_enc,cfg)
    if len(plan)!=HORIZON or plan_digest(plan)!=PLAN_SHA: raise ValueError('Original fit plan/order differs')
    runtime=build_d03_optimizer(model,len(plan),cfg,torch,device); validate_target(model,runtime)
    inputs={length:QwenInputs(tokenizer,length,(9693,2152)) for length in lengths}
    encoded={}; chunks={}; length_reports={'fit':fit_lengths}
    for length in lengths:
        for part,rows in [('dev',dev),('test',test)]:
            encoded[length,part],length_reports[f'{part}_{length}']=inputs[length].encode(rows,deadline)
            chunks[length,part]=eval_batches(rows,encoded[length,part],cfg['eval_batch'])
    return dict(torch=torch,device=device,model=model,runtime=runtime,fit=fit,fit_rows=fit_rows,
        fit_enc=fit_enc,train_inputs=train_inputs,plan=plan,work=work,inputs=inputs,
        encoded=encoded,chunks=chunks,dev=dev,test=test,sample=sample,
        fingerprints=fingerprints,lengths=length_reports)


def smoke_scoring(ctx):
    rng=capture_rng(ctx['torch'],ctx['device'].split(':')[0])
    before=parameter_digest(ctx['runtime'])
    rows=sorted(ctx['fit_rows'],key=lambda r:len(ctx['fit_enc'][r['id']]))
    chosen=[next(r for r in rows if int(r['relevance'])==y) for y in (0,1)]
    if len(ctx['fit_enc'][chosen[0]['id']])==len(ctx['fit_enc'][chosen[1]['id']]):
        chosen[1]=next(r for r in rows if len(ctx['fit_enc'][r['id']])!=len(ctx['fit_enc'][chosen[0]['id']]))
    try:
        result=checked_score_path(ctx['model'],ctx['train_inputs'],
            [(ctx['fit_enc'][r['id']],int(r['relevance'])) for r in chosen],ctx['torch'],ctx['device'])
        if parameter_digest(ctx['runtime'])!=before: raise ValueError('Score/backward probe modified parameters')
        return result
    finally:
        ctx['runtime'][1].zero_grad(set_to_none=True)
        restore_rng(rng,ctx['torch'],ctx['device'].split(':')[0])


def inference_profiles(ctx,cfg,deadline,report):
    runtime=ctx['runtime']; before=parameter_digest(runtime)
    rng=capture_rng(ctx['torch'],ctx['device'].split(':')[0]); profiles={}
    try:
        for length in ctx['inputs']:
            rows=profile_queries(ctx['dev'],ctx['encoded'][length,'dev'])
            enc=ctx['encoded'][length,'dev']; batches=eval_batches(rows,enc,cfg['eval_batch'])
            _,seconds=predict_batches(ctx['model'],batches,enc,ctx['inputs'][length],ctx['torch'],ctx['device'],deadline)
            observed=workload(batches,enc)
            profiles[length]={part:cfg['inference_time_margin']*projected_seconds(seconds,observed,
                workload(ctx['chunks'][length,part],ctx['encoded'][length,part])) for part in ('dev','test')}
            profiles[length].update(seconds=seconds,queries=len({r['query_id'] for r in rows}),rows=len(rows),
                workload=list(observed),**peak_memory(ctx['torch'],ctx['device']))
        if parameter_digest(runtime)!=before: raise ValueError('Inference modified parameters')
    finally: restore_rng(rng,ctx['torch'],ctx['device'].split(':')[0])
    report('profile',profiles=profiles,lengths=ctx['lengths'])
    return profiles


In [ ]:
def train_fixed_plan(plan, work, target, runtime, identity, checkpoint, previous,
                     train_deadline, cfg, torch, device, do_update, report):
    if not 0 < target <= len(plan):
        raise ValueError("Invalid fixed training target")
    step, progress = 0, {"rows": 0, "queries": 0}
    if previous is not None and (previous / "recovery.pt").is_file():
        step, progress = restore_d03_checkpoint(previous / "recovery.pt", runtime, identity, plan, torch, device)
        if step > target:
            raise ValueError("Recovery exceeds the frozen target")
    timings = []
    while step < target:
        left = train_deadline - time.monotonic()
        # Do not silently select a different training duration when time expires.
        if left <= 0 or (timings and left < 1.3 * fit_cost(timings, work[step:step + 1])):
            if step:
                save_d03_checkpoint(checkpoint, runtime, identity, step, progress, torch, device)
            raise BudgetExpired(f"Fixed target {target} not reached; saved cursor {step}")
        tick = time.monotonic()
        qids, rows = plan[step]
        rng_before = capture_rng(torch, device.split(":")[0])
        before = {n: p.detach().clone() for n, p in runtime[0].items() if "lora_B" in n} if step == 0 else {}
        try:
            stats = do_update(rows, train_deadline)
        except BudgetExpired:
            runtime[1].zero_grad(set_to_none=True)
            restore_rng(rng_before, torch, device.split(":")[0])
            if step:
                save_d03_checkpoint(checkpoint, runtime, identity, step, progress, torch, device)
            raise
        step += 1
        progress["rows"] += len(rows)
        progress["queries"] += len(qids)
        timings.append((time.monotonic() - tick, work[step - 1]))
        if before and not any(not torch.equal(old, runtime[0][n]) for n, old in before.items()):
            raise RuntimeError("The first update did not change LoRA B")
        if step == 1 or step % cfg["checkpoint_every"] == 0:
            save_d03_checkpoint(checkpoint, runtime, identity, step, progress, torch, device)
            if step == 1:
                step, progress = restore_d03_checkpoint(checkpoint, runtime, identity, plan, torch, device)
            report("fit", component=identity["component"], step=step, target=target,
                   update=stats, estimated_fit_seconds=fit_cost(timings, work[step:target]))
    save_d03_checkpoint(checkpoint, runtime, identity, step, progress, torch, device)
    # Full training state read-back, including optimizer/scheduler/scaler/RNG.
    step, progress = restore_d03_checkpoint(checkpoint, runtime, identity, plan, torch, device)
    return trained_progress(step, progress, len(plan), sum(len(rows) for _, rows in plan))


def continue_training(ctx,cfg,identity,recovery,target,deadline,report):
    timings=[]
    def update(rows,limit):
        cursor=ctx['runtime'][2].last_epoch
        if len(timings)>=cfg['profile_updates']:
            enough_time(limit,1.35*fit_cost(timings,ctx['work'][cursor:target]),'remaining fit trajectory')
        tick=time.monotonic(); lr=ctx['runtime'][1].param_groups[0]['lr']
        before=parameter_digest(ctx['runtime']) if not timings else None
        stats=fit_update(ctx['model'],rows,ctx['fit_enc'],ctx['train_inputs'],ctx['runtime'],cfg,ctx['torch'],ctx['device'],limit)
        if before is not None:
            if before==parameter_digest(ctx['runtime']): raise ValueError('First resumed update did not change LoRA')
            progress=dict(rows=sum(len(rs) for _,rs in ctx['plan'][:cursor+1]),
                          queries=sum(len(qs) for qs,_ in ctx['plan'][:cursor+1]))
            save_d03_checkpoint(recovery,ctx['runtime'],identity,cursor+1,progress,ctx['torch'],ctx['device'])
            restore_verified(recovery,ctx['runtime'],identity,ctx['plan'],cfg,ctx['torch'],ctx['device'])
            report('first_update_verified',step=cursor+1,learning_rate=lr,training=progress)
        timings.append((time.monotonic()-tick,ctx['work'][cursor]))
        return dict(stats,learning_rate=lr,update_number=cursor+1)
    result=train_fixed_plan(ctx['plan'],ctx['work'],target,ctx['runtime'],identity,recovery,
        recovery.parent if recovery.exists() else None,deadline,cfg,ctx['torch'],ctx['device'],update,report)
    restore_verified(recovery,ctx['runtime'],identity,ctx['plan'],cfg,ctx['torch'],ctx['device'])
    return result


## 10. Итоговые предсказания

Оцениваю все тестовые пары и записываю id,prediction.


In [ ]:
def read_scores(path):
    with Path(path).open(newline="", encoding="utf-8") as handle:
        reader = csv.reader(handle)
        if next(reader, None) != ["id", "prediction"]:
            raise ValueError("Expected id,prediction header")
        return [(row_id, float(score)) for row_id, score in reader]


def index_scores(pairs, expected_ids):
    indexed = {}
    for row_id, score in pairs:
        if row_id in indexed:
            raise ValueError("Duplicate prediction ID")
        indexed[row_id] = float(score)
    if indexed.keys() != expected_ids:
        raise ValueError("Prediction ID coverage mismatch")
    return indexed


def checked_scores(rows, predictions):
    ids = [row["id"] for row in rows]
    if not ids or len(set(ids)) != len(ids) or any(not isinstance(i, str) or not i for i in ids):
        raise ValueError("Expected unique nonempty string candidate IDs")
    expected = set(ids)
    scores = {}
    for row_id, value in predictions:
        if row_id in scores or row_id not in expected:
            raise ValueError("Duplicate or unknown prediction ID")
        scores[row_id] = float(value)
        if not isfinite(scores[row_id]):
            raise ValueError("Non-finite prediction")
    if scores.keys() != expected:
        raise ValueError("Missing prediction IDs")
    return scores


def write_submission(path, sample_ids, predictions):
    scores = index_scores(predictions, set(sample_ids))
    if not all(isfinite(value) for value in scores.values()):
        raise ValueError("Non-finite test prediction")
    temp = path.with_suffix(".tmp")
    try:
        with temp.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.writer(handle)
            writer.writerow(("id", "prediction"))
            writer.writerows((row_id, scores[row_id]) for row_id in sample_ids)
        if read_scores(temp) != [(row_id, scores[row_id]) for row_id in sample_ids]:
            raise ValueError("Submission did not round-trip with the expected IDs and scores")
        temp.replace(path)
    finally:
        temp.unlink(missing_ok=True)


def selected_test(ctx,cfg,identity,out,step,length,deadline,report):
    path=out/'artifacts'/f'step_{step}.pt'
    found,progress=restore_verified(path,ctx['runtime'],identity,ctx['plan'],cfg,ctx['torch'],ctx['device'])
    if found!=step: raise ValueError('Selected checkpoint cursor differs')
    before=parameter_digest(ctx['runtime']); report('test',selected_step=step,inference_length=length)
    pairs,seconds=predict_batches(ctx['model'],ctx['chunks'][length,'test'],ctx['encoded'][length,'test'],
        ctx['inputs'][length],ctx['torch'],ctx['device'],deadline)
    if parameter_digest(ctx['runtime'])!=before: raise ValueError('Test inference changed parameters')
    checked_scores(ctx['test'],pairs)
    write_submission(out/'submission.csv',ctx['sample'],pairs)
    dump_json(out/'test_complete.json',dict(identity=identity,step=step,inference_length=length,
        checkpoint_sha256=file_sha256(path),parameter_sha256=before,rows=len(pairs),
        submission_sha256=file_sha256(out/'submission.csv'),seconds=seconds,training=progress))
    report('complete',status='complete',selected_step=step,inference_length=length,
        submission_rows=len(pairs),submission_sha256=file_sha256(out/'submission.csv'),test_seconds=seconds)


In [ ]:
EARLY_STEP = 349


LENGTHS = (512, 768)


HISTORICAL_TEST_SECONDS = 2997.3197337910005


def run_autonomous_qwen(cfg,requirements,code,compute):
    """Fixed configuration only: official CSV + public pretrained -> fit -> test."""
    out,deadline,summary,report=new_run(cfg,code,compute); identity=None
    try:
        step=cfg['selected_step']; length=cfg['inference_length']
        if step not in (EARLY_STEP,HORIZON) or length not in LENGTHS: raise ValueError('Invalid fixed finalist')
        if cfg.get('source_dir') or cfg.get('resume_dir'): raise ValueError('Autonomous final starts from public pretrained')
        ctx=prepare_runtime(cfg,out,deadline,summary,requirements,[length])
        identity=execution_identity(cfg,code,compute,ctx['fingerprints'],summary['environment'],ctx['plan'])
        dump_json(out/'artifacts/identity.json',identity)
        floor=cfg['inference_time_margin']*HISTORICAL_TEST_SECONDS
        enough_time(deadline-cfg['save_reserve_seconds'],floor,'historical minimum test reserve')
        report('backward_probe',score_check=smoke_scoring(ctx))
        profiles=inference_profiles(ctx,cfg,deadline-cfg['save_reserve_seconds']-floor,report)
        recovery=out/'artifacts/recovery.pt'
        training=continue_training(ctx,cfg,identity,recovery,step,
            deadline-cfg['save_reserve_seconds']-profiles[length]['test'],report)
        shutil.copyfile(recovery,out/'artifacts'/f'step_{step}.pt')
        report('trained',training=training)
        selected_test(ctx,cfg,identity,out,step,length,deadline-cfg['save_reserve_seconds'],report)
    except Exception as exc:
        report('stopped',status='paused' if isinstance(exc,BudgetExpired) else 'failed',error=f'{type(exc).__name__}: {exc}')
        if not isinstance(exc,BudgetExpired): raise
    finally: transfer_manifest(out,identity)
    return summary


## 11. Запуск

Выполняю полный путь и проверяю готовый submission.csv.


In [ ]:
result = run_autonomous_qwen(CONFIG, REQUIREMENTS, CODE_SHA256, COMPUTE_SHA256)
if result["status"] != "complete":
    raise RuntimeError(result.get("error", "Неполный запуск"))
output = Path(CONFIG["output_root"]) / CONFIG["run_id"] / "submission.csv"
if result["submission_rows"] != OFFICIAL_TEST_ROWS or file_sha256(output) != result["submission_sha256"]:
    raise RuntimeError("Не совпали число строк или контрольная сумма submission.csv")
print(f"{output} · {result['submission_rows']} строк · SHA-256 {result['submission_sha256']}")
